#  🐍📊 NOTEBOOK DE TIMEO 🐍📊

## === GUIDE des fonctions du module local datalake === 

from DATALAKE.data import *

**data = data_download_gmd("country")** -> Telechargement des données liés à un pays via global_macro_dataset (BD publique)
**data = data_download_fred("indicator"** : str, "start" : str , "end" : str) -> Telechargement d'un indicateur via la FRED (BD publique)

**data_storing(data : dataframe, "nom_fichier" : str)** -> Range un dataframe "data" dans le DATALAKE en parquet, que vous venez de télécharger d'internet 

**data = import_parquet("file_name" : str)** -> importe un dataframe du DATALAKE dans votre file cible (notebook ou .py)

**which_parquet()** -> Vous renvoie une liste de l'ensemble des parquets dispo dans le DATALAKE

## === Pour importer les fonctions du fichier "outils_eda", si vous en avez besoin lors de votre analyse ===

import sys
import os

sys.path.append("../dorian_code") -> permet à Python d'aller lire les fichiers présent dans le dossier "dorian_code"

from outils_eda import * -> importe le fichier "outils_eda" dont ces fonctions que vous pouvez utiliser lors de l'analyse !

In [ ]:
from DATALAKE.data import *

# **⛓️⛓️ Recherches sur l'utilisation du Regime Switching Model de Markov ⛓️⛓️** 

## **<span style="color:red">I.     Principe**
Le modele à changement de regime de Markov permet de différencier l'effet d'une variables sur une autre en fonction d'états cachés nommés régimes étant endogène et donc non déterministe. Il apport une solution à nos modèles linéaires à travers la possibilité de modéliser la probabilité de passage entre les différents régimes traversés par les données.

Ce modèle permet d'aller au dela des modèles de changement structurels et permet surtout de mettre en avant la corrélation entre différentes variables dont l'effet évolue en fonction du temps. 


## **<span style="color:red">II.     Problème de notre étude lié directement aux régimes exogenes et discussion de la solution apportée par les chaines de Markov**
Dans les modèles précédents nous regressions de manière directe les rate sur nos différentes variables d'intèret en partant du principe que les threshold exogènes définis par le passage entre les différentes politiques monétaires, sont les réelles jonctions de l'effet observé en fonction des 3 régimes.

Empirquement, ceci peut il poser problème ? Oui. Admettons que les régimes que nous avons prédits comme éxogène et définie par les dates de passages entre les différentes politiques monétaires, nous ometterions par exemple que l'effet des chocs de politique monétaires change non pas uniquement en fonction des threshold donnés mais sont lagged de 5 ans pour le passage gold standard / bretton wood et seulement 2 ans pour le passage entre bretton woods et floating rate, nous sur(/sous)etimerions donc le beta propre aux différents régimes. Par exemple si l'effet des chocs de taux est supérieur durant la période bretton woods, si nous incluons dans l'estimation du beta de la période gold standard les 5 périodes lagged de passage au bretton woods, nous surestimerions le beta propre à la période du gold standard. 

Nous modélisons donc un modèle de passage de régimes endogène au sein de cette troisième partie pour contrôler pour ce risque de BVO propre aux trehsold exogenes. 

## **<span style="color:red">III.   Détail des limites des précédentes méthodes éxpérimentales** 
Pour justifier cette modélisation, mettons en avant les limites de chacune des précédentes méthodes éxperimentales solutionnés par le modèle  de Régime switching Markovien.

#### **<span style="color:green">Dummies regression : Au sein de cette partie nous partons du principe que les périodes des threshold correspondent aux annonces de transition des politiques monétaires.**

1. **Éxogénéité stricte des régimes et passage non dyamique** : si les passages entre les différents régimes sont lagged ou parfois causés par des chocs éxogènes (exemple : passage à bretton wood suite à la 1ere GM car le gold standard ne permettait pas de remonter les taux ), notre modèle peut omettre ce passage et donc perdre en significativité.

_*Markov*_ : Le passage entre les régimes devient endogène ce qui capte par exemple l'effet d'anticipation des périodes. 

2. **Effet constant dans le temps** : Au sein de ce modèle nous partons du principe que l'impact des taux sur les variables d'interets sous un régime donnée ne varie pas à l'intérieur du régime. Ceci peut être problématique car nous pouvons sur/sous éstimer ce même effet par exemple à cause d'une période outlier au sein du régime. Prenons par exemple la période floating rate, donc le beta des taux peut être dévalué à cause des données liés à la crise des subprime de 2008, la crise du COVID de 2021 ou encore homologuement la grande répression de 1930 pour la période Bretton Wood.

_*Markov*_ : Permet de capturer des changements de beta progressifs en fonction du temps et donc au sein de chaque régime, afin de contrôler pour ces "outliers effect" par exemple.

3. **Interactions inter-temporelle entre inflation, PIB et taux de changes** : Au sein de cette méthode de modélisation nous ne prenons pas en compte les chocs inter variables d'intérêts au sein de chaque régimes, potentiellement une source importante de variation que nous pourrions utiliser de 2 manières principales : 
        - Cela nous permettrait de modéliser l'éfficacité d'un régime de politique monétaire donné en minimisant la variances entre les différentes variables, signe de stabilisation de l'économie.
        - D'un autre sens, ce type de variation nous permettrait à contrario de mettre en avant que le cout associé à un régime donné n'est pas forcément du à l'innefficacité des taux d'interet pour stabiliser l'économie, mais aux variations trop élevés entre les variables du modèles au sein de ce régime. 

_*Markov*_ : Les effets éstimés par le MRS sont dynamique et donc peuvent prendre en compte comme au sein d'un modèle de Time Series comme VAR/SVAR, les interactions dynamiques entre les variables d'intérêts

#### **<span style="color:green">VAR / SVAR : Nous modélisons ici un modèle dynamique de variables (PIB, TI, inflation, taux changes) permettant de capturer les changements progressifs de l'effet des taux sur les variables d'intérêts, solutionnant en partie le points 1. et 3. précédents. Il part néanmoins de l'hypothèse que les régimes sont éxogènes et fixes.**

1. **Éxogénéité stricte des régimes** : Comme précédemment nous éstimons les transitions de manière éxogène. 

_*Markov*_ : Le passage entre les régimes devient endogène et stochastique ce qui capte par exemple l'effet d'anticipation des périodes. 

2. **Effet constant dans le temps** : Également comme précédement, les coefficients ne sont pas variables en fonction du temps. 

_*Markov*_ : Le passage entre les régimes devient endogène ce qui capte par exemple l'effet d'anticipation des périodes


#### **<span style="color:green"> Détails des atouts du modèle de régime Switching de Markov (MRS)**

1.	Les transitions sont endogènes : pas besoin d’annoncer comme fixe les changements de régime.
2.	Paramètres dynamiques par régime : chaque régime peut avoir ses propres coefficients pour le taux d’intérêt, inflation, PIB, etc.
3.	Captures les non-linéarités : l’effet des chocs économiques dépend du régime courant et n'est pas approximé comme une régression de manière linéaire.
4.	Persistances réalistes : peut modéliser la durée moyenne d’un régime via les probabilités de transition.
5.	MLE fournit des probabilités filtrées et lissées : on obtient non seulement la prévision endogène des régimes mais aussi la probabilité que l’économie soit dans ce régime donné.

_Question à se poser : Si on trouve qu'on à un effet de lag supérieur à des périodes données on pourra se demander si une des deux périodes n'entraine pas une certains inertie_

## **<span style="color:red"> IV. Méthode de modélisation** : 

Considérons donc 3 périodes tel que **$S_t \in \{{ 1, 3 , 2} \}$**. Nous suivrons les étapes suivantes :

#### 1. **Structure du modèle - MRS VAR** 

#### 2. **Spécification des régimes et calcul des probabilités de transition**

#### 3. **Réalisation de l'éstimation du modèle via MLE**

#### 4. **Extraction et interprétation des probabilités filtrés et lissées** 

#### 5. **Test, Validation et robustesse des résultats**

#### 6. **Interprétation**

#### 1. **Structure du modèle - MRS VAR** 

Nous partons ici du principe, en réutilisant les résultats significatif de la partie précédente, que chaque régime est décrit par un modèle VAR de variables d'intérêts ${i_t}$  et de régresseurs ${Y_t, \pi_t, e_t}$. 

Nous poserons pour chaque régime : $ Y_t = A_{S_t} \cdot Y_{t-1} + \epsilon_t $ avec une probabilité de passage $P(S_t | S_{t-1})$ 

La probabilité de passage entre chaque régimes suit ici une chaine de Markov. 

a. __**Chaine de Markov et justification de la propriété de mémoire courte**__

Une chaine de Markov est un processus stochastique dont l'état futur dépend de l'état présent et pas du passé complet, ce qu'on nomme la propriété de mémoire courte. 

Formellement, nous aurons que la probabilité d'être dans un régime à un instant donné dépend conditionnellement uniquement du régime de t-1 :

$ P(S_t | S_{t-1}, S_{t-2}, ...) = P(S_t | S_{t-1}) $ 

Intuitivement cela signifie dans notre cas que la probabilité d'être dans un régime donné à une instant t, bretton wood par exemple, dépend uniquement du régime précédent. Les régimes dans ce cadre ne sont pas observés mais impliqués par le modèle. Cette propriété nous permet donc d'éstimer la persistance d'un régime en fonction du temps en modélisant la probabilité de transitionner entre un régime donnée et un autre, ou rester dans un régime donné. Nous pourrons pour ilustrer ce processus, imaginer que la probabilité d'être dans le régime bretton wood, sachant que l'observation t-1 est une observation d'un régime bretton Wood, tend à décroitre en fonction du temps, jusqu'a laisser place au régime de floating rate. 

Nous pourrons nous poser la question du réalisme de prendre uniquement la période précédente. Vraisemblablement, nous partons du principe que l'histoire est indirectement contenu dans la période précédente, étant donné que le régime précédent peut être vue comme la finalitée d'une succession de régimes impliquant donc directement les probébilité d'appartenance à ces même. Prenons l'exemple d'un régime qui durerait longtemps, le gold standard en l'occurence, nous pouvons faire le postulat intuitif que la probabilité de rester au sein de ce régime peut tendre à une probabilité élevé d'y rester.

b. __**Probabilité de transition**__

Nous noterons un point d'attention à cette probabilité de transition. Dans un modèle de Markov simple, la probabilité de transitionner entre les différents régimes est supposée fixe et contenues dans la matrice de transition, une hypothèse nous permettant de réduire la complexité computationelle. Néanmoins les probabilité conditionelles d'être dans un régime donné à un instant t, dépendent elles du temps et sont donc variables. 
Cette hypothèse semble réaliste étant donnée que l'on part du principe que les changements d'institutions en économie sont lents et les régimes monétaires persistents. Cette approche n'est pas parfaite et peut faire place à des modèles plus avancés à probabilités variables non traités ici. 


#### 2. **Spécification des régimes et calcul des probabilités de transition**

Nous choisissons ici de forecaster 3 régimes distincts dans l'hypothèse que ces 3 régimes correspondent aux régimes que nous observons politiquement : {GOLD standard, Bretton Wood, Floating Rate}

a. __**Matrice de transition**__ 

Nous spécifions ensuite la matrice de transition markovienne ci jointe : $ P =
\begin{bmatrix}
p_{11} & p_{12} & p_{13} \\
p_{21} & p_{22} & p_{23} \\
p_{31} & p_{32} & p_{33}
\end{bmatrix} $

Avec $i,j \in {{Gold, BW, Float}}$ Nous pouvons estimer les coefficient de cette matrice de la manière suivante : 
- Diagonales de la matrice $P_{ii}$ : probabilité d'être dans un régime donné entre deux périodes 
- Transition de régime $P_{ij}$ : prenons pour exemple $i=Gold, \quad j = BW$, la probabilité $P_{Gold->BW}$ est la probabilité de transitionner d'un régime Gold Standard à un régime Bretton Wood. 

b. __**Calcul de la matrice de transition $P$**__

En éstimant par maximum likelihood estimation MLE les probabilités $ p_{ij} = Pr(S_t = j | S_{t-1} = i) \quad \forall{i,j}$